# Environment (Prediction of Future Values)

In [1]:
# Dynamic Environment
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd

class StockTradingEnv(gym.Env):
    metadata = {'render_modes': ['human'], 'render_fps': 30}

    def __init__(self, df, initial_balance=10000):
        super(StockTradingEnv, self).__init__()
        self.df = df
        self.initial_balance = initial_balance
        
        self.stock_price_history = df['Value'].values

        # Action Space: 0: Hold, 1: Buy, 2: Sell
        self.action_space = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)

        # Observation Space: (N closing prices, shares held, cash balance)
        # N=5, so shape is 5 + 2 = 7
        self.n = 24 * 7
        obs_len = self.n
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_len,), dtype=np.float32)

        self.current_step = max(0, self.n-1)
        
        # State tracking (will be reset in self.reset())
        self.balance = self.initial_balance
        self.shares_held = 0
        self.portfolio_value = self.initial_balance
        self.last_portfolio_value = self.initial_balance
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.balance = self.initial_balance
        self.shares_held = 0
        self.current_step = max(0, self.n-1) 
        self.last_portfolio_value = self.initial_balance
        
        # Get the initial observation (e.g., first n closing prices, 0 shares, initial balance)
        observation = self._get_observation().astype(np.float32)
        info = self._get_info()
        return observation, info

    def _get_observation(self):
        # A simple observation: last n close prices, shares held, current balance
        
        start = self.current_step - (self.n-1) if self.current_step >= (self.n-1) else 0
        prices = self.stock_price_history[start:self.current_step + 1]
        trend = ((prices[0] - prices[-1]) / prices[0]) * 100 # change percentage within window
        
        # Pad with zeros if less than n steps
        padded_prices = np.pad(prices, (self.n - len(prices), 0), 'constant', constant_values=prices[0])

        change = [] # change percentage between timesteps 
        for i in range(len(padded_prices)-1):
            change.append(((padded_prices[i] - padded_prices[i+1])/padded_prices[i])*100)
        
        return np.append(change, [trend]).astype(np.float32)

    def _get_info(self):
        # Calculate current total portfolio value
        current_price = self.stock_price_history[self.current_step]
        self.portfolio_value = self.balance + self.shares_held * current_price
        
        return {"portfolio_value": self.portfolio_value}
    
    def step(self, action, amount=0.001):
        self.current_step += 1
        
        current_price = self.stock_price_history[self.current_step]
        last_price = self.stock_price_history[self.current_step-1]

        pred_price = last_price * ((action**3)/20 + 1)
        
        # Calculate new portfolio value and reward
        new_portfolio_value = self.balance + self.shares_held * current_price
        reward = (abs(current_price - pred_price) / current_price) * -100
        self.last_portfolio_value = new_portfolio_value

        # Check for termination
        terminated = self.current_step >= len(self.stock_price_history) - 1
        truncated = False # No time limit termination
        
        observation = self._get_observation().astype(np.float32)
        info = self._get_info()
        
        return observation, float(reward.item()), terminated, truncated, info

    def render(self):
        # Simple print for demonstration
        print(f"Step: {self.current_step} | Price: {self.stock_price_history[self.current_step]:.2f} | Balance: {self.balance:.2f} | Shares: {self.shares_held} | Value: {self.portfolio_value:.2f}")

    def close(self):
        # Clean up resources if necessary
        pass

# You would then load your data and create an instance:
df = pd.read_csv('data_1766039871.csv')
#df = df.iloc[::-1].reset_index(drop=True)

split = int(df.shape[0] * 0.8)
split_1 = int(df.shape[0] * 0.95)

env = StockTradingEnv(df[:split])
env_test = StockTradingEnv(df[split:])

# Model

In [ ]:
# PPO w/ MLP Policy
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.logger import configure
import torch as th

timestep_model = None

checkpoint_callback = CheckpointCallback(
  save_freq=128000,              # How often to save (in timesteps)
  save_path="weights/pred_price",             # Where to save the model files
  name_prefix="ppo_stock_model"  # Prefix for the model files (e.g., ppo_stock_model_100000_steps.zip)
)

custom_arch = dict(net_arch=dict(pi=[512, 512], vf=[512, 512]))
policy_kwargs = dict(**custom_arch)

if timestep_model:
    model = PPO.load(f"weights/5min/ppo_stock_model_{timestep_model}_steps.zip",
                     env,
                     verbose=2, 
                     policy_kwargs=policy_kwargs,
                     n_steps=128,
                     learning_rate=1e-5)
else:
    model = PPO("MlpPolicy", 
            env, 
            verbose=2, 
            policy_kwargs=policy_kwargs,
            n_steps=128,
            learning_rate=3e-5)

model.learn(total_timesteps=10000000,
            callback=checkpoint_callback,
            log_interval=1000)

# Evaluation

In [5]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.logger import configure
import torch as th
from stable_baselines3.common.evaluation import evaluate_policy

# model: your trained SB3 agent (e.g., PPO, A2C, SAC instance)
# eval_env: A separate test environment (Gym/Gymnasium Env or VecEnv)

print("Test")

custom_arch = dict(net_arch=dict(pi=[512, 512], vf=[512, 512]))
policy_kwargs = dict(**custom_arch)

timestep_model = 2176000
model = PPO.load(f"weights/hour_pred_val/ppo_stock_model_{timestep_model}_steps.zip",
                 env,
                 verbose=2, 
                 policy_kwargs=policy_kwargs,
                 n_steps=128,
                 learning_rate=1e-5)

mean_reward, std_reward = evaluate_policy(
    model, 
    env_test, 
    n_eval_episodes=1, 
    deterministic=True
)

print(f"Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

Test
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Mean reward: -103.33 +/- 0.00


In [9]:
import gymnasium as gym
from stable_baselines3 import A2C # Or PPO, SAC, DQN, etc.
from stable_baselines3 import PPO

obs, info = env_test.reset()

timestep_model = 2176000
model = PPO.load(f"weights/hour_pred_val/ppo_stock_model_{timestep_model}_steps.zip")

# 3. Run the inference loop
differences = []
for i in range(1000):
    # Get the action from the model
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env_test.step(action)

    price = env_test.stock_price_history[env_test.current_step]
    pred_change_percentage = (action[0]**3)/20 + 1
    pred_price = env_test.stock_price_history[env_test.current_step-1] * pred_change_percentage
    
    print(f"{price:.2f}", '\t', 
          f"{pred_price:.2f}", '\t', 
          f"{price/env_test.stock_price_history[env_test.current_step-1]:.4f}",'\t',
          f"{pred_change_percentage:.4f}", '\t' )

    differences.append(price - pred_price)

89260.11 	 89126.55 	 1.0015 	 1.0000 	
89179.22 	 89251.19 	 0.9991 	 0.9999 	
89481.72 	 89172.35 	 1.0034 	 0.9999 	
89062.23 	 89463.34 	 0.9953 	 0.9998 	
88202.39 	 87967.64 	 0.9903 	 0.9877 	
89472.58 	 88219.22 	 1.0144 	 1.0002 	
89818.67 	 89480.79 	 1.0039 	 1.0001 	
90871.38 	 89478.59 	 1.0117 	 0.9962 	
91404.67 	 88933.01 	 1.0059 	 0.9787 	
91348.92 	 91446.14 	 0.9994 	 1.0005 	
91432.39 	 91350.41 	 1.0009 	 1.0000 	
90017.84 	 90309.01 	 0.9845 	 0.9877 	
89669.56 	 90005.97 	 0.9961 	 0.9999 	
90398.45 	 90112.69 	 1.0081 	 1.0049 	
90404.73 	 90399.00 	 1.0001 	 1.0000 	
91117.22 	 89857.97 	 1.0079 	 0.9940 	
91393.19 	 91119.46 	 1.0030 	 1.0000 	
91038.86 	 91573.19 	 0.9961 	 1.0020 	
91221.02 	 91315.65 	 1.0020 	 1.0030 	
91327.41 	 91221.77 	 1.0012 	 1.0000 	
91579.99 	 91327.35 	 1.0028 	 1.0000 	
91541.16 	 91642.80 	 0.9996 	 1.0007 	
91856.53 	 91541.20 	 1.0034 	 1.0000 	
91919.11 	 91725.10 	 1.0007 	 0.9986 	
92130.30 	 92018.77 	 1.0023 	 1.0011 	


IndexError: index 428 is out of bounds for axis 0 with size 428

In [10]:
np.mean(differences)

np.float64(25.321506853943383)

In [ ]:
from datetime import datetime, timedelta
